# Worked Capstone: Retail Sales EDA and Visual Story

**Domain:** Data analysis and business intelligence  
**Primary dataset:** `retail_sales.csv`  
**Level:** Practitioner to Advanced

## Business goal

Identify revenue and profit patterns, quantify data coverage, investigate discount and category behaviour, and deliver decision-oriented recommendations.

This is a worked reference project. First attempt the corresponding phase project independently; then use this capstone to compare framing, evaluation, code structure, and communication.

## Decision questions

        1. How do revenue and profit change over time?
2. Which categories and regions drive total profit and margin?
3. Where do discounts coincide with weak contribution?
4. Which findings are descriptive, and what would require an experiment?

        ## Definition of done

        - [ ] Quality audit
- [ ] KPI table
- [ ] Time trend
- [ ] Category/region analysis
- [ ] Relationship analysis
- [ ] Evidence-limitation-action summary

## End-to-end workflow

```text
Decision and scope
      ↓
Data contract and quality
      ↓
Exploration and hypotheses
      ↓
Baseline and evaluation design
      ↓
Candidate method(s)
      ↓
Held-out / temporal evaluation
      ↓
Error, slice, and sensitivity analysis
      ↓
Artifacts, limitations, recommendation
```

At every stage, distinguish calculation correctness, statistical validity, operational validity, and decision validity.

## Risk register

        | Risk | Mitigation |
        |---|---|
        | Observational association mistaken for causation | Use descriptive language and propose controlled tests. |
| Aggregate totals hide mix effects | Inspect margins and segment-level distributions. |
| Outliers dominate visual scale | Use robust summaries and disclose treatment. |

In [ ]:
from pathlib import Path
import sys
import json
import warnings
warnings.filterwarnings("ignore")

_candidates = [Path.cwd(), *Path.cwd().parents]
COURSE_ROOT = next((p for p in _candidates if (p / "datasets").exists()), Path.cwd())
DATA_DIR = COURSE_ROOT / "datasets"
ARTIFACT_DIR = COURSE_ROOT / "artifacts"
ARTIFACT_DIR.mkdir(exist_ok=True)
sys.path.insert(0, str(COURSE_ROOT))

import numpy as np
import pandas as pd
import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt
from IPython.display import display

RANDOM_SEED = 42
np.random.seed(RANDOM_SEED)
print(f"Course root: {COURSE_ROOT}")

## 1. Load, contract, and audit

Confirm row grain, types, time coverage, uniqueness, missingness, and financial identities before analysis.

In [ ]:
from src.course_utils import dataframe_audit
sales=pd.read_csv(DATA_DIR/"retail_sales.csv",parse_dates=["order_date"])
assert sales["order_id"].is_unique
assert (sales["units"]>0).all()
assert sales["discount_pct"].between(0,1).all()
identity_error=np.abs((sales["revenue"]-sales["cost"])-sales["profit"]).max()
assert identity_error < .02
display(dataframe_audit(sales))
print("Coverage:",sales.order_date.min(),sales.order_date.max(),"Rows:",len(sales))

## 2. KPIs and analytical grain

Calculate metrics at transaction grain, then explicitly aggregate to category and region.

In [ ]:
kpis=pd.Series({
    "orders":sales.order_id.nunique(),
    "units":sales.units.sum(),
    "revenue":sales.revenue.sum(),
    "profit":sales.profit.sum(),
    "margin":sales.profit.sum()/sales.revenue.sum(),
    "median_order_revenue":sales.revenue.median(),
})
display(kpis.to_frame("value"))
category=sales.groupby("category").agg(
    orders=("order_id","nunique"),units=("units","sum"),
    revenue=("revenue","sum"),profit=("profit","sum")
)
category["margin"]=category.profit/category.revenue
display(category.sort_values("profit",ascending=False).round(3))

## 3. Trend and seasonality

Use monthly aggregation for the long view and a trailing mean for short-term variability.

In [ ]:
monthly=sales.set_index("order_date").resample("MS").agg(revenue=("revenue","sum"),profit=("profit","sum"))
monthly["profit_margin"]=monthly.profit/monthly.revenue
fig,ax=plt.subplots(figsize=(10,4))
ax.plot(monthly.index,monthly.revenue,marker="o")
ax.set(title="Monthly retail revenue",xlabel="Month",ylabel="Revenue")
fig.autofmt_xdate(); plt.show()

fig,ax=plt.subplots(figsize=(10,4))
ax.plot(monthly.index,monthly.profit_margin,marker="o")
ax.set(title="Monthly profit margin",xlabel="Month",ylabel="Margin")
fig.autofmt_xdate(); plt.show()

## 4. Discount and profitability

Inspect association and segment summaries. Do not infer that changing discounts alone will cause the observed difference.

In [ ]:
sales["discount_band"]=pd.cut(sales.discount_pct,[-.001,.05,.10,.20,1],
                              labels=["0–5%","5–10%","10–20%",">20%"])
discount_summary=sales.groupby(["category","discount_band"],observed=True).agg(
    orders=("order_id","count"),mean_profit=("profit","mean"),
    median_profit=("profit","median"),mean_revenue=("revenue","mean")
).reset_index()
display(discount_summary.round(2))

fig,ax=plt.subplots(figsize=(7,4))
ax.scatter(sales.discount_pct,sales.profit,alpha=.25)
ax.set(title="Discount and order profit",xlabel="Discount rate",ylabel="Profit")
plt.show()

## 5. Decision summary artifact

Write findings as structured evidence rather than relying on notebook output.

In [ ]:
findings=pd.DataFrame([
    ["Profit concentration",category.profit.idxmax(),float(category.profit.max()),
     "Prioritize deeper driver analysis in the leading category."],
    ["Lowest margin category",category.margin.idxmin(),float(category.margin.min()),
     "Review cost and discount policy before pursuing volume growth."],
    ["Highest monthly revenue",str(monthly.revenue.idxmax().date()),float(monthly.revenue.max()),
     "Investigate campaign, seasonality, and mix before forecasting recurrence."],
],columns=["finding","segment_or_period","value","action"])
display(findings)
findings.to_csv(ARTIFACT_DIR/"capstone_retail_findings.csv",index=False)

## Model/project card

Complete this before presenting the result:

| Field | Statement |
|---|---|
| Intended use | |
| Excluded use | |
| Data population and coverage | |
| Target/metric definition | |
| Evaluation split | |
| Baseline | |
| Primary result | |
| Known limitations | |
| Important subgroup behaviour | |
| Human review / abstention | |
| Monitoring | |
| Owner and review cadence | |

## Final reflection

1. Which result changed your initial belief?
2. Which assumption creates the largest residual risk?
3. What simpler alternative was competitive?
4. What evidence is still required before an operational decision?
5. What would you monitor first after release?

Re-run the notebook from a clean kernel and verify generated artifacts before considering the capstone complete.